# micrograd — Neuron, Layer, MLP and training loop

Week 1, session 4. The `Value` class below is copied from `derivatives.ipynb` (commit 3320f3a).

In [36]:
import math

class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self._prev = set(_children)
        self.grad = 0.0
        self._backward = lambda: None
        self._op = _op

    def __add__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        new_value = self.data + other.data
        out =  Value(new_value,(self,other),"+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        value = self.data * other.data
        out = Value(value, (self,other), "*")

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward
        return out

    def __rmul__(self,other):
       return self * other

    def tanh(self):
        out = Value(math.tanh(self.data), (self,), "tanh")

        def _backward():
            self.grad += out.grad * (1 - math.tanh(self.data)**2)
        out._backward = _backward
        return out


    def backward(self):
        visited = set()
        topo = []
        def build(v):
            if v not in visited:
                for child in v._prev:
                    build(child)
                topo.append(v)
                visited.add(v)
        build(self)
        self.grad = 1.0
        for i in reversed(topo):
            i._backward()

    def __repr__(self):
        return f"Value={self.data}"


In [4]:
from viz import draw_dot

In [12]:
x1 = Value(1.0)
x2 = Value(2.0)
wA1 = Value(1.0)
wA2 = Value(1.0)
bA = Value(0.0)
iA = wA1*x1 + wA2*x2 + bA
oA = iA.tanh()

wB1 = Value(1.0)
wB2 = Value(1.0)
bB = Value(0.0)
iB = wB1*x1 + wB2*x2 + bB
oB = iB.tanh()

L = oA + oB
L.backward()

print(wA1.grad)
print(wB1.grad)

0.009866037165440211
0.009866037165440211


In [40]:
import random

class Neuron:
    def __init__(self, nin):
        self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1,1))

    def __call__(self, xs):
        s = Value(0.0)
        for xi, wi in zip(xs, self.weights):
            s += xi*wi
        s += self.bias
        return s.tanh()

n = Neuron(3)
n([5.0, 1.0, -3.0])


n = Neuron(3)
n.weights = [Value(1.0), Value(2.0), Value(-1.0)]
n.bias = Value(0.5)
n([0.5, -1.0, 1.0])
            

Value=-0.9640275800758169

In [38]:
print(Value(2.0) * 3.0)     
print(Value(2.0) + 1.0)

n = Neuron(3)
n.weights = [Value(1.0), Value(2.0), Value(-1.0)]
n.bias = Value(0.5)
o = n([0.5, -1.0, 1.0])
o.backward()
print([w.grad for w in n.weights], n.bias.grad)

Value=6.0
Value=3.0
[0.035325412426582214, -0.07065082485316443, 0.07065082485316443] 0.07065082485316443
